# EarningsLens — Stage 5: Hedge Scoring

Stage 5 computes a **hedge score** between 0.0 and 1.0 for every Specific FLS commitment sentence in the 2023–2025 scope. The hedge score quantifies how tentatively or confidently management phrased each commitment — a score near 0 indicates a direct, unqualified commitment; a score near 1 indicates a heavily hedged, uncertain statement.

The hedge score is one of the two primary signals in the EarningsLens credibility framework. It feeds directly into the credibility trend charts in the Stage 10 dashboard and into the cross-quarter tracking logic in Stage 6, where a rising hedge score on a repeated commitment is flagged as a potential credibility signal.

### Two-signal architecture

The hedge score combines two independent signals:

**Signal 1 — Loughran-McDonald (LM) hedge word frequency**  
The Loughran-McDonald Financial Sentiment Wordlist is the standard academic lexicon for financial NLP. Its `Uncertainty` and `Constraining` word categories capture hedge language specific to financial disclosure — words like *"approximately"*, *"subject to"*, *"may"*, *"intend"*, *"contingent"*. The LM score is the proportion of sentence tokens that appear in either category, normalised to 0–1.

**Signal 2 — finbert-tone confidence polarity**  
(`yiyanghkust/finbert-tone`) is a FinBERT model fine-tuned for financial tone classification. It outputs `positive`, `negative`, or `neutral` with a confidence probability. For hedge scoring, a `neutral` or `negative` tone with high confidence is mapped toward 1.0 (high hedge); a `positive` tone with high confidence is mapped toward 0.0 (low hedge). This captures tonal hedging that the LM wordlist misses — sentences phrased with positive confidence vs cautious uncertainty.

**Combined score**  
The two signals are blended with custom weighting into a single hedge score per sentence:
```
hedge_score = 0.65 × lm_score + 0.35 × tone_score
```
Both signals are independently meaningful — the blend is intentionally simple to keep the score interpretable.

### Output

The `hedge_score` column is written to `specific_fls_cache.parquet` alongside the existing slot filling columns. Supabase writes remain deferred and will be pushed as a single batch before the dashboard build.

## 0. Dependencies

`transformers` and `torch` are already installed from Stage 3. The Loughran-McDonald wordlist is downloaded directly from the author's website as a CSV — it is freely available for academic use and does not require a HuggingFace model download. `requests` handles the download.

In [ ]:
# %pip install -q transformers torch tqdm requests pandas

## 0.1 Imports & Device Configuration

CUDA is confirmed before model loading. The same RTX 4060 used in Stage 3 will handle the `finbert-tone` inference pass — at batch size 64 and float16, 93k sentences processes in approximately 10-15 minutes.

In [ ]:
# Standard library
import os
import re
import logging
import zipfile
from pathlib import Path
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv
import hashlib


# Deep learning
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Data
import pandas as pd
import requests
from tqdm import tqdm

c:\Users\sidsu\Desktop\Sem 4\NLP\NLPvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
logging.basicConfig(
    format="%(asctime)s [%(levelname)s] %(message)s",
    level=logging.INFO,
)
log = logging.getLogger("earningslens.stage5")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA not available. Stage 5 requires a GPU for finbert-tone inference."
    )

In [ ]:
DEVICE = torch.device("cuda")
log.info("CUDA device : %s", torch.cuda.get_device_name(0))
log.info("VRAM        : %.1f GB", torch.cuda.get_device_properties(0).total_memory / 1e9)
print("✓ Device configured")

2026-05-06 22:45:38,054 [INFO] CUDA device : NVIDIA GeForce RTX 4060 Laptop GPU
2026-05-06 22:45:38,056 [INFO] VRAM        : 8.6 GB


✓ Device configured


In [ ]:
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
if not DATABASE_URL:
    raise EnvironmentError("DATABASE_URL not found — check your .env file")

## 1. Configuration

`LM_BLEND` and `TONE_BLEND` control the weighting of each signal in the final hedge score. Equal weighting is the default — adjustable here if post-hoc analysis shows one signal is more discriminative for this corpus. The two weights must sum to 1.0.

In [ ]:
# Model
TONE_MODEL_NAME = "yiyanghkust/finbert-tone"
BATCH_SIZE      = 64     # reduce to 32 if CUDA OOM
MAX_LENGTH      = 128

# Hedge score blend weights — must sum to 1.0
LM_BLEND   = 0.65    # Loughran-McDonald lexical hedge signal weight
TONE_BLEND = 0.35    # finbert-tone confidence polarity signal weight

# Year scope — consistent with Stage 4
YEAR_FILTER = list(range(2023, 2026))

# Paths
INPUT_PATH  = Path("specific_fls_cache.parquet")
LM_PATH     = Path("Loughran-McDonald_MasterDictionary_1993-2025.csv")

print("✓ Configuration set")
print(f"  Model       : {TONE_MODEL_NAME}")
print(f"  Blend       : LM={LM_BLEND}  tone={TONE_BLEND}")
print(f"  Year scope  : {YEAR_FILTER[0]}–{YEAR_FILTER[-1]}")

✓ Configuration set
  Model       : yiyanghkust/finbert-tone
  Blend       : LM=0.65  tone=0.35
  Year scope  : 2023–2025


In [ ]:
# Custom financial hedge phrases — covers modal verbs, qualifiers, and
# conditional language that dominate earnings call guidance sentences
# These supplement the LM wordlist which is sparse on modal hedge language

CUSTOM_HEDGE_PHRASES = {
    # Modal verbs
    "may", "might", "could", "would", "should",
    # Forward-looking verbs with uncertainty
    "expect", "expects", "expected", "expecting",
    "believe", "believes", "believed",
    "estimate", "estimates", "estimated",
    "forecast", "forecasts", "forecasted",
    "project", "projects", "projected",
    "intend", "intends", "intended",
    # Qualifiers
    "approximately", "roughly", "around", "about",
    "potential", "potentially",
    "possible", "possibly",
    "likely", "unlikely",
    "tend", "tends",
    "assume", "assumes", "assumed", "assuming",
    # Conditional
    "subject to", "contingent", "dependent",
    "if", "provided that", "assuming that",
    # Range language
    "range", "between", "up to", "at least",
    "more than", "less than", "above", "below",
    # Softeners
    "somewhat", "slightly", "modestly", "broadly",
    "generally", "typically", "usually",
}

## 2. Load Loughran-McDonald Wordlist

The Loughran-McDonald Master Dictionary is downloaded from the author's server at Notre Dame. It contains sentiment category flags for approximately 86,000 financial words. The `Uncertainty` and `Constraining` columns are the relevant categories for hedge scoring — a non-zero value in either column marks the word as hedge language in financial contexts.

Words are lowercased and stored in a set for O(1) lookup during sentence scoring.

In [ ]:
# Load and build hedge word set from Uncertainty and Constraining categories
lm_df = pd.read_csv(LM_PATH, low_memory=False)

# Uncertainty and Constraining columns contain year-of-first-use integers
# when flagged, and 0 when not — non-zero means the word is in that category
hedge_words = set(
    lm_df[
        (lm_df["Uncertainty"].fillna(0).astype(int) != 0) |
        (lm_df["Constraining"].fillna(0).astype(int) != 0)
    ]["Word"].str.lower().tolist()
)

print(f"LM hedge word set size : {len(hedge_words):,} words")
print(f"Sample hedge words     : {list(hedge_words)[:10]}")

LM hedge word set size : 476 words
Sample hedge words     : ['reexamination', 'turbulence', 'permission', 'encumbrances', 'earmark', 'possible', 'anticipate', 'randomizing', 'restraining', 'escrow']


## 3. Load Input Data

`specific_fls_cache.parquet` is loaded and filtered to the 2023–2025 scope. This is the same set of sentences processed in Stage 4, now carrying `metric`, `value`, and `timeframe` columns from the slot filling stage.

In [ ]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"{INPUT_PATH} not found. "
        "Ensure Stage 4 completed and updated specific_fls_cache.parquet."
    )

In [ ]:
fls = pd.read_parquet(INPUT_PATH)
log.info("Loaded %d rows from %s", len(fls), INPUT_PATH)

2026-05-06 22:45:48,696 [INFO] Loaded 92458 rows from specific_fls_cache.parquet


In [ ]:
log.info("Feature columns: {}".format(fls.columns.to_list()))

2026-05-06 22:45:50,372 [INFO] Feature columns: ['transcript_id', 'ticker', 'company_name', 'quarter', 'year', 'speaker', 'speaker_role', 'segment_type', 'sentence_index', 'sentence', 'fls_label', 'fls_confidence', 'sentence_hash', 'metric', 'value', 'timeframe']


## 4. Loughran-McDonald Hedge Scoring — Signal 1

Each sentence is tokenised by splitting on whitespace and punctuation, lowercased, and matched against the LM hedge word set. The LM score is the fraction of tokens that are hedge words, clipped to a maximum of 0.2 before normalisation to 0–1.

The 0.2 clip prevents runaway scores on very short sentences where a single hedge word would otherwise produce a disproportionately high score. In practice, financial guidance sentences rarely exceed 15% hedge word density — a sentence at the 0.2 clip is already extremely heavily hedged.

This pass runs on CPU and processes 93k sentences in under a minute.

In [ ]:
# Extend the hedge_words set with custom phrases
hedge_words_extended = hedge_words | CUSTOM_HEDGE_PHRASES
print(f"Original hedge set  : {len(hedge_words):,}")
print(f"Extended hedge set  : {len(hedge_words_extended):,}")
print(f"Custom additions    : {len(hedge_words_extended) - len(hedge_words):,}")

Original hedge set  : 476
Extended hedge set  : 520
Custom additions    : 44


In [ ]:
# Regex tokeniser — splits on whitespace and punctuation, preserves words
TOKEN_RE = re.compile(r"[a-zA-Z']+")

# Maximum hedge word density before normalisation clip
# Sentences above this density are all mapped to LM score = 1.0
LM_CLIP = 0.20

In [ ]:
def lm_hedge_score(sentence: str) -> float:
    """
    Compute the Loughran-McDonald hedge score for one sentence.

    Counts the proportion of tokens that appear in the LM Uncertainty
    or Constraining word categories, then normalises to 0–1 with a
    density clip at LM_CLIP to prevent short-sentence distortion.

    Args:
        sentence : Raw sentence string

    Returns:
        Float in [0.0, 1.0] — higher means more hedge language
    """
    tokens = TOKEN_RE.findall(sentence.lower())
    if not tokens:
        return 0.0

    hedge_count = sum(1 for t in tokens if t in hedge_words_extended)
    raw_density = hedge_count / len(tokens)

    # Normalise: raw_density / LM_CLIP, clipped to [0, 1]
    return min(raw_density / LM_CLIP, 1.0)

In [ ]:
log.info("Computing LM hedge scores for %d sentences...", len(fls))
fls["lm_score"] = fls["sentence"].apply(lm_hedge_score)

print("LM score distribution:")
print(fls["lm_score"].describe().round(3).to_string())
print(f"\nSentences with lm_score = 0   : {(fls['lm_score'] == 0).sum():,}  (no hedge words found)")
print(f"Sentences with lm_score = 1   : {(fls['lm_score'] == 1).sum():,}  (at or above clip threshold)")

2026-05-06 22:46:03,404 [INFO] Computing LM hedge scores for 92458 sentences...


LM score distribution:
count    92458.000
mean         0.269
std          0.242
min          0.000
25%          0.000
50%          0.238
75%          0.405
max          1.000

Sentences with lm_score = 0   : 25,023  (no hedge words found)
Sentences with lm_score = 1   : 1,441  (at or above clip threshold)


In [ ]:
# Measure actual LM hit rate in the corpus
hit_rate = (fls["lm_score"] > 0).mean() * 100
mean_when_hit = fls[fls["lm_score"] > 0]["lm_score"].mean()

print(f"Sentences with any LM hedge word : {hit_rate:.1f}%")
print(f"Mean lm_score when > 0           : {mean_when_hit:.3f}")
print(f"Mean lm_score overall            : {fls['lm_score'].mean():.3f}")

Sentences with any LM hedge word : 72.9%
Mean lm_score when > 0           : 0.370
Mean lm_score overall            : 0.269


## 5. Load finbert-tone — Signal 2

`yiyanghkust/finbert-tone` classifies each sentence as `positive`, `negative`, or `neutral` with a softmax probability. For hedge scoring, the tone signal is computed as:

- `positive` tone → tone_score = `1 - confidence` (confident positive = low hedge)
- `neutral` tone → tone_score = `0.5` (ambiguous, moderate hedge)
- `negative` tone → tone_score = `confidence` (confident negative = high hedge)

This mapping reflects the intuition that management expressing confident positive expectations is making a stronger commitment than management expressing neutral or negative hedged outlooks.

The model is loaded in float16 to halve VRAM usage. On the RTX 4060 with batch size 64, 200k sentences completes in approximately 20–30 minutes.

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

log.info("Loading finbert-tone...")

# finbert-tone predates the AutoModel config requirements — load with explicit
# BertForSequenceClassification and BertTokenizer from bert-base-uncased
tone_tokeniser = BertTokenizer.from_pretrained("bert-base-uncased")

tone_model = BertForSequenceClassification.from_pretrained(
    TONE_MODEL_NAME,
    torch_dtype=torch.float16,
    ignore_mismatched_sizes=True,
)
tone_model.to(DEVICE)
tone_model.eval()

log.info("Label mapping : %s", tone_model.config.id2label)
print("✓ finbert-tone loaded")
print(f"  Labels : {tone_model.config.id2label}")

2026-05-06 22:46:07,120 [INFO] Loading finbert-tone...
2026-05-06 22:46:07,627 [INFO] HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-06 22:46:07,887 [INFO] HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-05-06 22:46:08,122 [INFO] HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-06 22:46:08,359 [INFO] HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-05-06 22:46:08,615 [INFO] HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-05-06 22:46:08,917 [INFO] HTTP Request: HEAD https:

✓ finbert-tone loaded
  Labels : {0: 'Neutral', 1: 'Positive', 2: 'Negative'}


2026-05-06 22:46:10,805 [INFO] HTTP Request: GET https://huggingface.co/api/models/yiyanghkust/finbert-tone/discussions?p=0 "HTTP/1.1 200 OK"
2026-05-06 22:46:11,059 [INFO] HTTP Request: GET https://huggingface.co/api/models/yiyanghkust/finbert-tone/commits/refs%2Fpr%2F18 "HTTP/1.1 200 OK"
2026-05-06 22:46:11,294 [INFO] HTTP Request: HEAD https://huggingface.co/yiyanghkust/finbert-tone/resolve/refs%2Fpr%2F18/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-05-06 22:46:11,531 [INFO] HTTP Request: HEAD https://huggingface.co/yiyanghkust/finbert-tone/resolve/refs%2Fpr%2F18/model.safetensors "HTTP/1.1 302 Found"


## 6. finbert-tone Inference — Signal 2

The tone model runs over all sentences in batches. The tone score per sentence is derived from the predicted label and its softmax confidence using the mapping described above. Results are stored in `tone_score` alongside the intermediate `tone_label` and `tone_confidence` columns for interpretability.

In [ ]:
def tone_to_hedge(label: str, confidence: float) -> float:
    """
    Convert a finbert-tone label and confidence into a hedge score contribution.

    positive + high confidence → low hedge (approaching 0.0)
    neutral                    → moderate hedge (0.5)
    negative + high confidence → high hedge (approaching 1.0)

    Args:
        label      : 'positive', 'neutral', or 'negative'
        confidence : softmax probability of the predicted label

    Returns:
        Float in [0.0, 1.0]
    """
    label = label.lower()
    if label == "positive":
        return 1.0 - confidence   # high confidence positive → near 0
    elif label == "negative":
        return confidence          # high confidence negative → near 1
    else:                          # neutral
        return 0.5

In [ ]:
def run_tone_batch(texts: list[str]) -> tuple[list[str], list[float], list[float]]:
    """
    Run one batch of sentences through finbert-tone.

    Returns:
        labels      : Predicted tone label per sentence
        confidences : Softmax probability of predicted label
        tone_scores : Derived hedge contribution per sentence
    """
    encoded = tone_tokeniser(
        texts,
        padding    = True,
        truncation = True,
        max_length = MAX_LENGTH,
        return_tensors = "pt",
    ).to(DEVICE)

    with torch.inference_mode():
        logits = tone_model(**encoded).logits

    probs      = torch.softmax(logits, dim=-1).cpu()
    pred_ids   = probs.argmax(dim=-1).tolist()
    pred_probs = probs.max(dim=-1).values.tolist()
    labels     = [tone_model.config.id2label[i] for i in pred_ids]
    scores     = [tone_to_hedge(l, c) for l, c in zip(labels, pred_probs)]

    return labels, pred_probs, scores

In [ ]:
# ── Inference loop ────────────────────────────────────────────────────────────
sentences_list  = fls["sentence"].tolist()
all_labels      = []
all_confidences = []
all_tone_scores = []

In [ ]:

log.info(
    "Running finbert-tone over %d sentences ",
    len(sentences_list)
)

for batch_start in tqdm(range(0, len(sentences_list), BATCH_SIZE), desc="finbert-tone"):
    batch = sentences_list[batch_start : batch_start + BATCH_SIZE]
    labels, confidences, scores = run_tone_batch(batch)
    all_labels.extend(labels)
    all_confidences.extend(confidences)
    all_tone_scores.extend(scores)

fls["tone_label"]      = all_labels
fls["tone_confidence"] = all_confidences
fls["tone_score"]      = all_tone_scores

2026-05-06 22:46:14,354 [INFO] Running finbert-tone over 92458 sentences 
finbert-tone: 100%|██████████| 1445/1445 [01:01<00:00, 23.32it/s]


In [ ]:
log.info("finbert-tone inference complete")
print("\nTone label distribution:")
print(fls["tone_label"].value_counts().to_string())
print(f"\nTone score distribution:")
print(fls["tone_score"].describe().round(3).to_string())

2026-05-06 22:47:16,340 [INFO] finbert-tone inference complete



Tone label distribution:
tone_label
Neutral     88492
Positive     3059
Negative      907

Tone score distribution:
count    92458.000
mean         0.491
std          0.079
min          0.000
25%          0.500
50%          0.500
75%          0.500
max          1.000


## 7. Combine Signals into Final Hedge Score

The LM score and tone score are blended using the weights defined in the configuration cell. The final `hedge_score` is clipped to [0.0, 1.0] to handle any floating point edge cases from the blend arithmetic.

In [ ]:
# Blend the two signals into the final hedge score
# hedge_score = LM_BLEND × lm_score + TONE_BLEND × tone_score
fls["hedge_score"] = (
    LM_BLEND   * fls["lm_score"] +
    TONE_BLEND * fls["tone_score"]
).clip(0.0, 1.0)

print("Final hedge score distribution:")
print(fls["hedge_score"].describe().round(3).to_string())

Final hedge score distribution:
count    92458.000
mean         0.347
std          0.160
min          0.000
25%          0.175
50%          0.323
75%          0.435
max          1.000


In [ ]:
# Bucket distribution for a quick sanity check
print("\nBucket distribution:")
buckets = pd.cut(
    fls["hedge_score"],
    bins  = [0, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels= ["0.0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"]
).value_counts().sort_index()

for bucket, count in buckets.items():
    pct = count / len(fls) * 100
    bar = "█" * int(pct / 2)
    print(f"  {bucket} : {count:>7,}  ({pct:4.1f}%)  {bar}")


Bucket distribution:
  0.0-0.2 :  25,175  (27.2%)  █████████████
  0.2-0.4 :  37,786  (40.9%)  ████████████████████
  0.4-0.6 :  22,145  (24.0%)  ███████████
  0.6-0.8 :   5,766  ( 6.2%)  ███
  0.8-1.0 :   1,431  ( 1.5%)  


## 8. Validation — High and Low Hedge Examples

Qualitative validation of the hedge score is done by inspecting the highest and lowest scoring sentences. Low-scoring sentences should be direct, confident, quantified commitments. High-scoring sentences should contain qualifying language, conditions, or uncertain phrasing. This check confirms the two signals are combining as intended before the results are written to disk.

In [ ]:
print("── Lowest hedge scores (most confident commitments) ──")
low_hedge = fls.nsmallest(8, "hedge_score")[["sentence", "hedge_score", "tone_label", "lm_score"]]
for _, row in low_hedge.iterrows():
    print(f"  [{row['hedge_score']:.3f} | tone={row['tone_label']} | lm={row['lm_score']:.3f}]")
    print(f"  {str(row['sentence'])[:130]}")
    print()

── Lowest hedge scores (most confident commitments) ──
  [0.000 | tone=Positive | lm=0.000]
  And we remain on track to begin a Phase 3 study later this year in third-line CRC.

  [0.000 | tone=Positive | lm=0.000]
  We'll add $1 billion of revenue this year, add 1 million users.

  [0.000 | tone=Positive | lm=0.000]
  This work will increase the effectiveness and efficiency of messaging to create more impactful experiences.

  [0.000 | tone=Positive | lm=0.000]
  We are looking forward to continuing our leadership and momentum into 2024.

  [0.000 | tone=Positive | lm=0.000]
  So the hundreds of millions of monthly active users of Reader will also be able to get access to AI Assistant and purchase an add-

  [0.000 | tone=Positive | lm=0.000]
  We’ll begin to deploy our wireless solution in their next-gen EVs in 2026.

  [0.000 | tone=Positive | lm=0.000]
  This business has come a long way since its launch in 2008, and we look forward to leaning into our outsourcing business as a dif

In [ ]:
print("── Highest hedge scores (most hedged statements) ──")
high_hedge = fls.nlargest(8, "hedge_score")[["sentence", "hedge_score", "tone_label", "lm_score"]]
for _, row in high_hedge.iterrows():
    print(f"  [{row['hedge_score']:.3f} | tone={row['tone_label']} | lm={row['lm_score']:.3f}]")
    print(f"  {str(row['sentence'])[:130]}")
    print()

── Highest hedge scores (most hedged statements) ──
  [1.000 | tone=Negative | lm=1.000]
  We expect about half of this to install in 2024.

  [1.000 | tone=Negative | lm=1.000]
  Gross margin for the full year should be about 57.5%.

  [1.000 | tone=Negative | lm=1.000]
  I would anticipate we'd take price again in fiscal Q1.

  [1.000 | tone=Negative | lm=1.000]
  For Q2, we expect SAP expense to be about flat.

  [1.000 | tone=Negative | lm=1.000]
  We're expecting revenues of about $6.1 billion.

  [0.999 | tone=Negative | lm=1.000]
  It would be for the remaining 4 projects of the 8.

  [0.999 | tone=Negative | lm=1.000]
  I'd expect about 50% of that flows through.

  [0.999 | tone=Negative | lm=1.000]
  And at this point, we're expecting it to be about 65.50%.



## 9. Hedge Score by Speaker Role & Segment

The hedge score distribution is broken down by speaker role and segment type to validate that the signal behaves as expected across the corpus structure. Executives in prepared remarks are expected to have lower mean hedge scores than analysts in Q&A — executives typically deliver polished, pre-prepared guidance while Q&A responses are more hedged and conditional.

In [ ]:
print("── Mean hedge score by speaker role ──")
role_hedge = fls.groupby("speaker_role")["hedge_score"].agg(["mean", "median", "count"])
role_hedge.columns = ["mean", "median", "count"]
role_hedge = role_hedge.sort_values("mean")
print(role_hedge.round(3).to_string())

── Mean hedge score by speaker role ──
               mean  median  count
speaker_role                      
analyst       0.326   0.308  22002
executive     0.354   0.330  70456


In [ ]:
print("\n── Mean hedge score by segment type ──")
seg_hedge = fls.groupby("segment_type")["hedge_score"].agg(["mean", "median", "count"])
seg_hedge.columns = ["mean", "median", "count"]
print(seg_hedge.round(3).to_string())


── Mean hedge score by segment type ──
                   mean  median  count
segment_type                          
prepared_remarks  0.354   0.330  70528
qa                0.326   0.308  21930


In [ ]:
print("\n── Mean hedge score by year ──")
year_hedge = fls.groupby("year")["hedge_score"].agg(["mean", "median", "count"])
year_hedge.columns = ["mean", "median", "count"]
print(year_hedge.round(3).to_string())


── Mean hedge score by year ──
       mean  median  count
year                      
2023  0.347   0.323  42632
2024  0.347   0.323  37782
2025  0.348   0.326  12044


## 10. Write Results - Local Parquet and Supabase

The `hedge_score`, `lm_score`, `tone_label`, `tone_confidence`, and `tone_score` columns are merged back into the full `specific_fls_cache.parquet` and Supabase via `sentence_hash`. Rows outside the 2023–2025 scope retain their existing columns unchanged — only the filtered subset receives the new hedge columns.

`specific_fls_cache.parquet` now contains the complete output of Stages 3, 4, and 5 and is the input to Stage 6 (cross-quarter tracking).

In [ ]:
WORKING_PATH = Path("fls_2023_2025.parquet")

fls.to_parquet(WORKING_PATH, index=False)

print("✓ fls_2023_2025.parquet written")
print(f"  Total rows              : {len(fls):,}")
print(f"  Rows with hedge_score   : {fls['hedge_score'].notna().sum():,}")
print(f"  Columns                 : {list(fls.columns)}")

✓ fls_2023_2025.parquet written
  Total rows              : 92,458
  Rows with hedge_score   : 92,458
  Columns                 : ['transcript_id', 'ticker', 'company_name', 'quarter', 'year', 'speaker', 'speaker_role', 'segment_type', 'sentence_index', 'sentence', 'fls_label', 'fls_confidence', 'sentence_hash', 'metric', 'value', 'timeframe', 'lm_score', 'tone_label', 'tone_confidence', 'tone_score', 'hedge_score']


In [ ]:
conn   = psycopg2.connect(DATABASE_URL + "?sslmode=require")
cursor = conn.cursor()

hedge_rows = [
    (float(row["hedge_score"]), row["sentence_hash"])
    for _, row in fls.iterrows()
    if pd.notna(row["hedge_score"]) and pd.notna(row["sentence_hash"])
]

print(f"Rows to update : {len(hedge_rows):,}")

CHUNK = 2000
for i in range(0, len(hedge_rows), CHUNK):
    execute_values(
        cursor,
        """
        UPDATE commitments
        SET hedge_score = data.hedge_score
        FROM (VALUES %s) AS data(hedge_score, sentence_hash)
        WHERE commitments.sentence_hash = data.sentence_hash
        """,
        hedge_rows[i:i+CHUNK],
        page_size=2000,
    )
    conn.commit()
    print(f"✓ {min(i+CHUNK, len(hedge_rows)):,} / {len(hedge_rows):,} rows committed")

cursor.close()
conn.close()
print(f"\n✓ Stage 5 — {len(hedge_rows):,} hedge scores written to Supabase")

Rows to update : 92,458
✓ 2,000 / 92,458 rows committed
✓ 4,000 / 92,458 rows committed
✓ 6,000 / 92,458 rows committed
✓ 8,000 / 92,458 rows committed
✓ 10,000 / 92,458 rows committed
✓ 12,000 / 92,458 rows committed
✓ 14,000 / 92,458 rows committed
✓ 16,000 / 92,458 rows committed
✓ 18,000 / 92,458 rows committed
✓ 20,000 / 92,458 rows committed
✓ 22,000 / 92,458 rows committed
✓ 24,000 / 92,458 rows committed
✓ 26,000 / 92,458 rows committed
✓ 28,000 / 92,458 rows committed
✓ 30,000 / 92,458 rows committed
✓ 32,000 / 92,458 rows committed
✓ 34,000 / 92,458 rows committed
✓ 36,000 / 92,458 rows committed
✓ 38,000 / 92,458 rows committed
✓ 40,000 / 92,458 rows committed
✓ 42,000 / 92,458 rows committed
✓ 44,000 / 92,458 rows committed
✓ 46,000 / 92,458 rows committed
✓ 48,000 / 92,458 rows committed
✓ 50,000 / 92,458 rows committed
✓ 52,000 / 92,458 rows committed
✓ 54,000 / 92,458 rows committed
✓ 56,000 / 92,458 rows committed
✓ 58,000 / 92,458 rows committed
✓ 60,000 / 92,458 rows 

## Stage 5 Summary

Hedge scoring is complete for the 2023–2025 commitment corpus.

In [ ]:
print("=" * 54)
print("  EarningsLens — Stage 5 Complete")
print("=" * 54)
print(f"\nSentences scored         : {len(fls):,}")
print(f"Mean hedge score         : {fls['hedge_score'].mean():.3f}")
print(f"Median hedge score       : {fls['hedge_score'].median():.3f}")
print(f"Std dev                  : {fls['hedge_score'].std():.3f}")
print(f"\nSignal breakdown:")
print(f"  LM mean score          : {fls['lm_score'].mean():.3f}")
print(f"  Tone mean score        : {fls['tone_score'].mean():.3f}")
print(f"\nOutput written to        : {WORKING_PATH.resolve()}")
print(f"\n→ Ready for Stage 6 — Cross-Quarter Commitment Tracking")

  EarningsLens — Stage 5 Complete

Sentences scored         : 92,458
Mean hedge score         : 0.347
Median hedge score       : 0.323
Std dev                  : 0.160

Signal breakdown:
  LM mean score          : 0.269
  Tone mean score        : 0.491

Output written to        : C:\Users\sidsu\Desktop\Sem 4\NLP\AT2\fls_2023_2025.parquet

→ Ready for Stage 6 — Cross-Quarter Commitment Tracking


## Handoff to Stage 6

Stage 5 is complete. The outputs are:

| Output | Location | Consumed by |
|---|---|---|
| `fls_2023_2025.parquet` | Local disk | Stage 6 — cross-quarter tracking |

The cache now carries the full pipeline output for 2023–2025 Specific FLS sentences: `sentence_hash`, `fls_label`, `fls_confidence`, `metric`, `value`, `timeframe`, `hedge_score`, `lm_score`, `tone_label`, `tone_confidence`, `tone_score`.

Stage 6 uses `all-MiniLM-L6-v2` sentence embeddings and cosine similarity to match commitments across quarters for the same company — identifying when management is making the same commitment in different words across multiple earnings calls. Each matched pair is assigned a status of `Delivered`, `Raised`, `Missed`, or `Withdrawn` by comparing the extracted `value` slots across quarters. This is the cross-quarter credibility intelligence that differentiates EarningsLens from existing financial data products.